In [74]:
import warnings
warnings.filterwarnings('ignore')

import lazypredict
from lazypredict.Supervised import LazyRegressor



import yfinance as yf
import pandas as pd
from sklearn.model_selection import train_test_split
#from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt
import numpy as np

In [75]:
end_date = '2024-10-15'
stock_data = yf.download('^GSPC', start='2019-10-15', end=end_date)

[*********************100%***********************]  1 of 1 completed


In [76]:
#Download additional data for 10 days after the initial end date

additional_data = yf.download('^GSPC', start=pd.to_datetime(end_date) + pd.Timedelta(days=1), 
end=pd.to_datetime(end_date) + pd.Timedelta(days=10))

[*********************100%***********************]  1 of 1 completed


In [77]:
# Step 2: Feature Engineering
# Calculate daily returns
stock_data['Return'] = stock_data['Adj Close'].pct_change()
additional_data['Return'] = additional_data['Adj Close'].pct_change()

In [78]:
# Calculate rolling volatility (standard deviation of returns over a window)
stock_data['Volatility'] = stock_data['Return'].rolling(window=21).std()  # 21-day rolling volatility
additional_data['Volatility'] = additional_data['Return'].rolling(window=21).std()

In [79]:
# Drop NaN values
stock_data.dropna(inplace=True)
additional_data.dropna(inplace=True)

# Step 3: Feature Selection
# We use lagged returns and volatility as features
stock_data['Lagged_Volatility'] = stock_data['Volatility'].shift(1)
stock_data['Lagged_Return'] = stock_data['Return'].shift(1)
additional_data['Lagged_Volatility'] = additional_data['Volatility'].shift(1)
additional_data['Lagged_Return'] = additional_data['Return'].shift(1)


In [80]:
# Prepare the features and target for modeling
X = stock_data[['Lagged_Volatility', 'Lagged_Return']]
y = stock_data['Volatility']

# Drop NaN values that result from shifting
X = X.dropna()
y = y.loc[X.index]

In [81]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False)

# Initialize LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)

# Fit LazyRegressor on the training data and evaluate on the test data
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

# Print the results
print(models)

100%|██████████| 42/42 [00:02<00:00, 14.63it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000119 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 865, number of used features: 2
[LightGBM] [Info] Start training from score 0.012621
                               Adjusted R-Squared  R-Squared  RMSE  Time Taken
Model                                                                         
HuberRegressor                               0.96       0.96  0.00        0.03
RANSACRegressor                              0.96       0.96  0.00        0.02
OrthogonalMatchingPursuitCV                  0.96       0.96  0.00        0.01
OrthogonalMatchingPursuit                    0.96       0.96  0.00        0.01
SGDRegressor                                 0.96       0.96  0.00        0.01
LassoCV                                      

In [82]:
# Step 4: Model Training using Gradient Boosting Regressor
gbr_model = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.1, max_depth=3, random_state=42)
gbr_model.fit(X_train, y_train)

# Step 5: Volatility Prediction and Evaluation
y_pred = gbr_model.predict(X_test)
mse = root_mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error for Volatility Prediction: {mse}')


Mean Squared Error for Volatility Prediction: 0.0005176542077394631


In [83]:
from sklearn.linear_model import HuberRegressor

# Step 4: Model Training using Huber Regressor
huber_model = HuberRegressor()
huber_model.fit(X_train, y_train)

# Step 5: Volatility Prediction and Evaluation
y_pred_ = huber_model.predict(X_test)
mse = root_mean_squared_error(y_test, y_pred_)
print(f'Mean Squared Error for Volatility Prediction: {mse}')

Mean Squared Error for Volatility Prediction: 0.0004046821748768276


In [84]:
import warnings
warnings.filterwarnings('ignore')

# Step 6: Forecasting Future Volatility
forecast_horizon = 10  # Number of days to forecast
forecasted_volatility = []
forecasted_volatility_ = []
forecast_dates = pd.date_range(start=X_test.index[-1] + pd.Timedelta(days=1), periods=forecast_horizon, freq='B')

last_X = X_test.iloc[-1, :].values.reshape(1, -1)  # Take the last row from X_test

for _ in range(forecast_horizon):
    # Predict the next volatility
    next_volatility = gbr_model.predict(last_X)
    next_volatility_=huber_model.predict(last_X)

    forecasted_volatility.append(next_volatility[0])
    forecasted_volatility_.append(next_volatility_[0])

    # Update last_X with the predicted volatility and simulate a small random return
    new_lagged_volatility = next_volatility[0]
    new_lagged_return = np.random.normal(0, np.std(X_test['Lagged_Return']))  # Simulate return with noise
    last_X = np.array([[new_lagged_volatility, new_lagged_return]])

# Convert forecasted data to DataFrame for easy plotting
forecast_df = pd.DataFrame({
    'Date': forecast_dates,
    'Forecasted_Volatility': forecasted_volatility
})

forecast_df_ = pd.DataFrame({
    'Date': forecast_dates,
    'Forecasted_Volatility Huber': forecasted_volatility_
})


In [85]:
# True data plot steps
end_date_true_plot = '2024-10-26'
stock_data_true = yf.download('^GSPC', start='2023-01-01', end=end_date_true_plot)
stock_data_true['Return'] = stock_data_true['Adj Close'].pct_change()
stock_data_true['Volatility'] = stock_data_true['Return'].rolling(window=21).std()
stock_data_true.dropna(inplace=True)


[*********************100%***********************]  1 of 1 completed


In [87]:
from plotly.subplots import make_subplots
import plotly.graph_objs as go

# Create a figure
fig = make_subplots()

# Add predicted volatility trace
fig.add_trace(go.Scatter(
    x=y_test.index, 
    y=y_pred * 100,  # Convert to percentage
    mode='lines', 
    name='Gradient Boosting Regressor', 
    line=dict(color='green', dash='solid'),
    opacity=0.5
))

# Add forecasted volatility trace
fig.add_trace(go.Scatter(
    x=forecast_df['Date'], 
    y=forecast_df['Forecasted_Volatility'] * 100,  # Convert to percentage
    mode='lines', 
    name='Forecasted Volatility-Gradient Boosting', 
    line=dict(color='red', dash='dash'),
    opacity=0.7
))

# Add predicted volatility trace
fig.add_trace(go.Scatter(
    x=y_test.index, 
    y=y_pred_ * 100,  # Convert to percentage
    mode='lines', 
    name='Huber Regressor', 
    line=dict(color='orange', dash='solid'),
    opacity=0.5
))

# Add forecasted volatility trace
fig.add_trace(go.Scatter(
    x=forecast_df_['Date'], 
    y=forecast_df_['Forecasted_Volatility Huber'] * 100,  # Convert to percentage
    mode='lines', 
    name='Forecasted Volatility - Huber', 
    line=dict(color='black', dash='dash'),
    opacity=0.7
))

# Add actual volatility trace
fig.add_trace(go.Scatter(
    x=stock_data_true.index, 
    y=stock_data_true['Volatility'] * 100,  # Convert to percentage
    mode='lines', 
    name='Actual Volatility', 
    line=dict(color='blue', dash='solid')
))

# Update layout
fig.update_layout(
    title='SP500 Stock Volatility: Actual, Predicted, and Forecast',
    xaxis_title='Date',
    yaxis_title='Volatility (%)',  # Update Y axis title
    legend=dict(x=0, y=1)
)

# Show the figure
fig.show()
